# Exhaustive Timeframe Combination Backtest — 100 Days

Sweeps **every** multi-timeframe gate combo (all 13 TFs, subsets up to 3) × entry TF (M1/M5/M15/M30/H1) × SL:TP (1.5/2.0/3.0) × SL-ATR (1.5/2.0/3.0) × exits (full/partial) × filters (none/energy/zone/energy+zone), time-aligned, no lookahead, spread-adjusted.

- Data: parquet export from local MT5 gateway (5 FX majors, 9 native TFs; H2/H3/H6/H8/H12 resampled from M30)
- Engine: exact port of research_v2.py `sim()` — same signals, gates, partial exits
- Gates shifted 1 bar, reindexed by timestamp, ffilled (no intrabar lookahead)
- GPU not needed for the sweep (numpy vectorized inner loop); GPU used for the Monte Carlo bootstrap tail

Upload the `colab_export` folder (zipped) to Drive at `/content/drive/MyDrive/mt5_ea/colab_export.zip`.

In [ ]:
# 1. Setup: mount Drive, unpack data
import os, json, time
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/colab_export'
ZIP = '/content/drive/MyDrive/mt5_ea/colab_export.zip'
OUT_DIR = '/content/drive/MyDrive/mt5_ea/sweep_results'
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    import zipfile
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content')
print('data files:', len([f for f in os.listdir(DATA_DIR) if f.endswith('.parquet')]))
print('GPU check:')
import torch
print('  cuda:', torch.cuda.is_available())

In [ ]:
# 2. Load native TFs + resample the missing ones from M30
SYMBOLS = ['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD', 'NZDUSD']
PIP = {s: (0.01 if 'JPY' in s else 0.0001) for s in SYMBOLS}
SPREAD_PIPS = {'EURUSD': 1.0, 'GBPUSD': 1.4, 'USDJPY': 1.2, 'AUDUSD': 1.3, 'NZDUSD': 1.8}

NATIVE_TFS = ['M1', 'M5', 'M15', 'M30', 'H1', 'H4', 'D1', 'W1', 'MN1']
RESAMPLE_MAP = {'H2': '2h', 'H3': '3h', 'H6': '6h', 'H8': '8h', 'H12': '12h'}
ALL_TFS = NATIVE_TFS + list(RESAMPLE_MAP)

def load_tf(sym, tf):
    p = f'{DATA_DIR}/{sym}_{tf}.parquet'
    if os.path.exists(p):
        df = pd.read_parquet(p)
    elif tf in RESAMPLE_MAP:
        m30 = pd.read_parquet(f'{DATA_DIR}/{sym}_M30.parquet')
        m30['time'] = pd.to_datetime(m30['time'])
        m30 = m30.set_index('time')
        rule = RESAMPLE_MAP[tf]
        df = m30.resample(rule, label='left', closed='left').agg(
            {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
             'tick_volume': 'sum', 'spread': 'mean', 'real_volume': 'sum'}).dropna().reset_index()
        df['time'] = df['time'].astype(str)
    else:
        return None
    return df.sort_values('time').reset_index(drop=True)

DATA = {s: {} for s in SYMBOLS}
for s in SYMBOLS:
    for tf in ALL_TFS:
        df = load_tf(s, tf)
        if df is not None and len(df) > 60:
            DATA[s][tf] = df
    print(s, {tf: len(d) for tf, d in DATA[s].items()})

In [ ]:
# 3. Engine: bias + alignment + vectorized sim (exact port of research_v2)
def ema(s, w): return s.ewm(span=w, adjust=False).mean()

def atr_df(df, p=14):
    h, l, c = df['high'].values, df['low'].values, df['close'].values
    prev = np.roll(c, 1); prev[0] = c[0]
    tr = np.maximum(h - l, np.maximum(np.abs(h - prev), np.abs(l - prev)))
    out = pd.Series(tr).rolling(p).mean().values
    return out

def bias_np(df, fast=50, slow=200):
    """+1 bull / -1 bear / 0 neutral, computed on df's own bars; returns (times, bias)."""
    c = df['close'].values
    ef = pd.Series(c).ewm(span=fast, adjust=False).mean().values
    es = pd.Series(c).ewm(span=slow, adjust=False).mean().values
    t = np.zeros(len(c))
    t[ef > es] = 1; t[ef < es] = -1
    rh = pd.Series(df['high'].values).rolling(20).max().shift(20).values
    rl = pd.Series(df['low'].values).rolling(20).min().shift(20).values
    close = c
    with np.errstate(invalid='ignore'):
        bull = (t == 1) & (close > rh * 0.999)
        bear = (t == -1) & (close < rl * 1.001)
    b = np.where(bull, 1, np.where(bear, -1, 0))
    # neutral where EMAs disagree with structure signal stays 0 unless breakout confirms
    b = np.where(t == 1, np.where(bull, 1, 0), b)
    b = np.where(t == -1, np.where(bear, -1, 0), b)
    return df['time'].values, b

def align_gate(gate_times, gate_vals, h1_times):
    """Shift gate 1 bar, then ffill onto H1 timestamps."""
    gs = pd.Series(gate_vals[1:], index=gate_times[1:])  # shift(1), drop first bar
    aligned = gs.reindex(h1_times, method='ffill')
    return aligned.fillna(0).values

def sim(h1_close, h1_high, h1_low, gates, sl_tp, use_energy, use_zone,
        pv, start, sl_atr, partial, spread_pips, a, atr_avg, rh20, rl20):
    """Vectorized-friendly sim: same logic as research_v2.sim but numpy inner loop."""
    n = len(h1_close)
    ef = pd.Series(h1_close).ewm(span=50, adjust=False).mean().values
    es = pd.Series(h1_close).ewm(span=200, adjust=False).mean().values
    trades = []
    in_pos = 0; ei = 0; entry = sl = tp = 0.0
    p_taken = False; pnl_taken = 0.0; p_frac = 0.0; p_level = 0.0
    for i in range(start, n - 1):
        p = h1_close[i]
        if in_pos == 0:
            if ef[i] > es[i]: sig = 1
            elif ef[i] < es[i]: sig = -1
            else: continue
            ok = True
            for g in gates:
                if g[i] != sig: ok = False; break
            if not ok: continue
            if use_energy and not np.isnan(atr_avg[i]) and atr_avg[i] > 0:
                if a[i] / atr_avg[i] < 1.5: continue
            if use_zone:
                za = a[i] if not np.isnan(a[i]) else 0.001
                if sig == 1 and p > rl20[i] + 2 * za: continue
                if sig == -1 and p < rh20[i] - 2 * za: continue
            entry = p; stop = max(sl_atr * a[i], 1e-4)
            sl = entry - sig * stop; tp = entry + sig * sl_tp * stop
            in_pos = sig; ei = i; p_taken = False; pnl_taken = 0.0
            if partial:
                p_frac = partial['frac']; p_level = entry + sig * partial['r'] * stop
            else:
                p_frac = 0.0
        else:
            hi, lo = h1_high[i], h1_low[i]
            if partial and not p_taken:
                if (in_pos == 1 and hi >= p_level) or (in_pos == -1 and lo <= p_level):
                    pnl_taken = ((p_level - entry) if in_pos == 1 else (entry - p_level)) / pv * p_frac
                    p_taken = True
            exit_pnl = None
            if in_pos == 1:
                if lo <= sl: exit_pnl = (sl - entry) / pv
                elif hi >= tp: exit_pnl = (tp - entry) / pv
            else:
                if hi >= sl: exit_pnl = (entry - sl) / pv
                elif lo <= tp: exit_pnl = (entry - tp) / pv
            if exit_pnl is not None:
                remaining = 1.0 - (p_frac if partial else 0.0)
                trades.append(exit_pnl * remaining + pnl_taken - spread_pips)
                in_pos = 0
            elif in_pos != 0 and i - ei > 200:
                pnl = ((p - entry) if in_pos == 1 else (entry - p)) / pv
                remaining = 1.0 - (p_frac if partial else 0.0)
                trades.append(pnl * remaining + pnl_taken - spread_pips)
                in_pos = 0
    return trades

def stats(trades):
    n = len(trades)
    if n == 0: return {'trades': 0, 'win%': 0, 'expect': 0, 'PF': 0, 'sharpe': 0, 'max_dd': 0}
    p = np.array(trades)
    wins = p[p > 0].sum(); losses = abs(p[p <= 0].sum())
    sharpe = float(p.mean() / p.std() * np.sqrt(252)) if p.std() > 0 else 0
    eq = np.cumsum(p); peak = np.maximum.accumulate(eq)
    return {'trades': n, 'win%': round(float((p > 0).mean() * 100), 1),
            'expect': round(float(p.mean()), 4),
            'PF': round(float(wins / losses), 2) if losses else 0,
            'sharpe': round(sharpe, 2), 'max_dd': round(float((peak - eq).max()), 2)}

print('engine ready')

In [ ]:
# 4. Build the sweep space: every gate subset up to 3 TFs
from itertools import combinations

ENTRY_TFS = ['M1', 'M5', 'M15', 'M30', 'H1']
GATE_POOL = ['M5', 'M15', 'M30', 'H1', 'H2', 'H3', 'H4', 'H6', 'H8', 'H12', 'D1', 'W1', 'MN1']

gate_combos = {'none': []}
for r in (1, 2, 3):
    for combo in combinations(GATE_POOL, r):
        gate_combos['+'.join(combo)] = list(combo)
print(f'gate subsets: {len(gate_combos)} (singletons: {len(GATE_POOL)}, pairs: {sum(1 for _ in combinations(GATE_POOL,2))}, triples: {sum(1 for _ in combinations(GATE_POOL,3))})')

SL_TP = [1.5, 2.0, 3.0]
SL_ATR = [1.5, 2.0, 3.0]
EXITS = [(None, 'full'), ({'r': 1.0, 'frac': 0.5}, 'partial1R')]
FILTERS = [(False, False, 'none'), (True, False, 'energy'), (False, True, 'zone'), (True, True, 'energy+zone')]

total = len(ENTRY_TFS) * len(gate_combos) * len(SL_TP) * len(SL_ATR) * len(EXITS) * len(FILTERS)
print(f'total combos: {total:,}')
print(f'  (entry 5) x (gates {len(gate_combos)}) x (sltp 3) x (slatr 3) x (exit 2) x (filt 4)')

In [ ]:
# 5. Precompute per-symbol arrays, then RUN THE FULL SWEEP
t0 = time.time()

# Precompute H1 sim-base arrays + all gate alignments per symbol
PRE = {}
for s in SYMBOLS:
    h1 = DATA[s]['H1']
    h1_times = pd.Index(pd.to_datetime(h1['time']).values)
    pre = {
        'h1_times': h1_times,
        'close': h1['close'].values.astype(float),
        'high': h1['high'].values.astype(float),
        'low': h1['low'].values.astype(float),
        'atr': atr_df(h1),
        'pip': PIP[s],
        'spread': SPREAD_PIPS.get(s, 1.5),
        'gates': {},
    }
    pre['atr_avg'] = pd.Series(pre['atr']).rolling(50).mean().values
    pre['rh20'] = pd.Series(pre['high']).rolling(20).max().values
    pre['rl20'] = pd.Series(pre['low']).rolling(20).min().values
    # entry-TF 'gates' are just their own aligned bias
    for tf in set(GATE_POOL) | set(ENTRY_TFS):
        if tf not in DATA[s]:
            continue
        gt, gv = bias_np(DATA[s][tf])
        pre['gates'][tf] = align_gate(gt, gv, h1_times)
        pre[f'first_real_{tf}'] = gt[np.nonzero(gv)[0][0]] if (gv != 0).any() else None
    PRE[s] = pre
print(f'precompute done in {time.time()-t0:.0f}s')

results = []
done = 0
t0 = time.time()
checkpoint_every = 20000

for etf in ENTRY_TFS:
    for gn, gate_tfs in gate_combos.items():
        # per-symbol gate availability filter
        for st in SL_TP:
            for sa in SL_ATR:
                for partial, ex_name in EXITS:
                    for ue, uz, fn in FILTERS:
                        all_trades = []
                        for s in SYMBOLS:
                            pre = PRE[s]
                            if etf not in pre['gates']:
                                continue
                            # every gate tf must exist for this symbol
                            if any(g not in pre['gates'] for g in gate_tfs):
                                continue
                            # start after all gates have real data + 250 warmup
                            starts = []
                            for g in [etf] + gate_tfs:
                                fr = pre.get(f'first_real_{g}')
                                if fr is not None:
                                    idx = pre['h1_times'].searchsorted(pd.Timestamp(fr))
                                    if idx < len(pre['h1_times']):
                                        starts.append(idx)
                            start_i = (max(starts) + 250) if starts else 250
                            gates = [pre['gates'][etf]] + [pre['gates'][g] for g in gate_tfs]
                            tt = sim(pre['close'], pre['high'], pre['low'], gates,
                                     st, ue, uz, pre['pip'], start_i, sa,
                                     partial, pre['spread'], pre['atr'],
                                     pre['atr_avg'], pre['rh20'], pre['rl20'])
                            all_trades.extend(tt)
                        stt = stats(all_trades)
                        results.append({'entry_tf': etf, 'gates': gn, 'sl_tp': st,
                                         'sl_atr': sa, 'exit': ex_name, 'filters': fn, **stt})
                        done += 1
                        if done % checkpoint_every == 0:
                            df_partial = pd.DataFrame(results)
                            df_partial.to_json(f'{OUT_DIR}/sweep_checkpoint.json', orient='records', lines=True)
                            print(f'  {done:,}/{total:,} ({time.time()-t0:.0f}s)', flush=True)

df = pd.DataFrame(results)
df.to_json(f'{OUT_DIR}/sweep_full_{time.strftime("%Y%m%d")}.json', orient='records', lines=True)
df.to_parquet(f'{OUT_DIR}/sweep_full_{time.strftime("%Y%m%d")}.parquet', index=False)
print(f'\nSWEEP DONE: {len(results):,} combos in {time.time()-t0:.0f}s')
df.head()

In [ ]:
# 6. Analysis: robust rankings (min 50 trades) + gate-shape economics
robust = df[df['trades'] >= 50]
print(f'robust combos (>=50 trades): {len(robust):,} / {len(df):,}')

print('\n=== TOP 25 ROBUST COMBOS ===')
cols = ['entry_tf', 'gates', 'sl_tp', 'sl_atr', 'exit', 'filters', 'trades', 'win%', 'expect', 'PF', 'sharpe', 'max_dd']
top = robust.sort_values('expect', ascending=False).head(25)
print(top[cols].to_string(index=False))

print('\n=== EXPECTANCY BY GATE CARDINALITY ===')
df['gate_n'] = df['gates'].apply(lambda g: 0 if g == 'none' else g.count('+') + 1)
print(df.groupby('gate_n').agg(avg_exp=('expect', 'mean'), avg_pf=('PF', 'mean'),
                                total_trades=('trades', 'sum')).round(2))

print('\n=== EXPECTANCY BY ENTRY TF ===')
print(df.groupby('entry_tf').agg(avg_exp=('expect', 'mean'), avg_pf=('PF', 'mean'),
                                 total_trades=('trades', 'sum')).round(2))

print('\n=== TOP SINGLE-TF GATES (cardinality 1, >=50 trades) ===')
single = robust[robust['gates'] != 'none'].copy()
single['g1'] = single['gates']
print(single[single['gates'].str.count('\\+') == 0].groupby('g1').agg(
    avg_exp=('expect', 'mean'), best_exp=('expect', 'max'),
    total_trades=('trades', 'sum')).round(2).sort_values('avg_exp', ascending=False))

print('\n=== TOP PAIR GATES ===')
pairs = robust[robust['gates'].str.count('\\+') == 1]
print(pairs.groupby('gates').agg(avg_exp=('expect', 'mean'), best_exp=('expect', 'max'),
    total_trades=('trades', 'sum')).round(2).sort_values('avg_exp', ascending=False).head(15))

print('\n=== W1 / D1 / H12 / H8 CONTAINMENT (do big TFs help anywhere?) ===')
for tf in ['W1', 'D1', 'H12', 'H8', 'H6', 'H3', 'H2']:
    with_tf = robust[robust['gates'].str.contains(tf)]
    without = robust[~robust['gates'].str.contains(tf)]
    print(f'{tf:>4}: with avg_exp={with_tf["expect"].mean():.2f} ({len(with_tf)} combos) | '
          f'without avg_exp={without["expect"].mean():.2f}')

# save the analysis tables
top[cols].to_csv(f'{OUT_DIR}/top25_robust.csv', index=False)
print(f'\nsaved: {OUT_DIR}/top25_robust.csv')

In [ ]:
# 7. GPU Monte Carlo bootstrap on top-20 robust combos (torch)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

top20 = robust.sort_values('expect', ascending=False).head(20)
N_BOOT = 20000

boot_rows = []
for _, row in top20.iterrows():
    # re-run the sim for this exact combo to get raw trade pnls
    all_trades = []
    for s in SYMBOLS:
        pre = PRE[s]
        etf = row['entry_tf']
        gate_tfs = [] if row['gates'] == 'none' else row['gates'].split('+')
        if etf not in pre['gates']: continue
        if any(g not in pre['gates'] for g in gate_tfs): continue
        starts = []
        for g in [etf] + gate_tfs:
            fr = pre.get(f'first_real_{g}')
            if fr is not None:
                idx = pre['h1_times'].searchsorted(pd.Timestamp(fr))
                if idx < len(pre['h1_times']): starts.append(idx)
        start_i = (max(starts) + 250) if starts else 250
        gates = [pre['gates'][etf]] + [pre['gates'][g] for g in gate_tfs]
        partial = {'r': 1.0, 'frac': 0.5} if row['exit'] == 'partial1R' else None
        ue = 'energy' in row['filters']; uz = 'zone' in row['filters']
        tt = sim(pre['close'], pre['high'], pre['low'], gates, row['sl_tp'], ue, uz,
                 pre['pip'], start_i, row['sl_atr'], partial, pre['spread'],
                 pre['atr'], pre['atr_avg'], pre['rh20'], pre['rl20'])
        all_trades.extend(tt)
    if len(all_trades) < 10: continue
    p = torch.tensor(all_trades, dtype=torch.float32, device=device)
    n = len(p)
    idx = torch.randint(0, n, (N_BOOT, n), device=device)
    samples = p[idx]
    exps = samples.mean(dim=1)
    wins = torch.clamp(samples, min=0).sum(dim=1)
    losses = torch.clamp(samples, max=0).abs().sum(dim=1)
    pfs = torch.where(losses > 0, wins / losses, torch.tensor(float('inf'), device=device))
    boot_rows.append({
        'entry_tf': row['entry_tf'], 'gates': row['gates'], 'sl_tp': row['sl_tp'],
        'sl_atr': row['sl_atr'], 'exit': row['exit'], 'filters': row['filters'],
        'trades': n,
        'exp_p5': round(float(exps.quantile(0.05)), 4),
        'exp_p50': round(float(exps.quantile(0.5)), 4),
        'pf_p5': round(float(pfs.quantile(0.05)), 2),
        'prob_profit': round(float((exps > 0).float().mean()), 3),
    })
    print(f"{row['entry_tf']} {row['gates']}: exp_p5={boot_rows[-1]['exp_p5']}, pf_p5={boot_rows[-1]['pf_p5']}, P(profit)={boot_rows[-1]['prob_profit']}")

boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(f'{OUT_DIR}/bootstrap_top20.csv', index=False)
print('\nsaved:', f'{OUT_DIR}/bootstrap_top20.csv')
boot_df